# Agent Playbooks & Improvement

> **Time:** ~15 minutes | **Level:** Intermediate

Reflexio extracts two types of learning from agent conversations:
- **User Profiles** — facts about the user (covered in notebook 02)
- **Agent Playbooks** — corrections and improvements for the agent

This notebook focuses on the playbook pipeline:

```
Interactions → [Extract] → User Playbook → [Aggregate] → Agent Playbook
```

### Prerequisites
- Reflexio server running (`uv run reflexio services start --only backend`)
- `OPENAI_API_KEY` set in your `.env` file

In [1]:
import uuid

from _display_helpers import *

from reflexio import (
    InteractionData,
    PlaybookAggregatorConfig,
    PlaybookConfig,
    PlaybookStatus,
    ReflexioClient,
    UserPlaybook,
)

RUN_ID = uuid.uuid4().hex[:8]
USER_ID = f"playbook_demo_{RUN_ID}"

url, api_key = load_env()
client = ReflexioClient(url_endpoint=url, api_key=api_key)

# Configure playbook extraction and aggregation for this notebook
config = client.get_config()

OK Environment loaded — server URL: http://localhost:8081

## Seed: A Conversation with Agent Mistakes

> We publish a **single conversation** where the agent makes mistakes. Reflexio will auto-extract user playbook entries from it. Later, we'll add more entries manually to demonstrate aggregation.

In [2]:
response = client.publish_interaction(
    user_id=USER_ID,
    interactions=[
        InteractionData(role="User", content="Hi, I bought an Acme ProBook laptop about three weeks ago and I'd like to return it. Is it too late?"),
        InteractionData(role="Agent", content="Unfortunately, our return window is only 14 days from the date of purchase, so I'm afraid your return period has expired. I'm sorry about that!"),
        InteractionData(role="User", content="Wait, that doesn't sound right. Your website clearly says 30 days for electronics. Can you double-check?"),
        InteractionData(role="Agent", content="You're absolutely right, I apologize! Our return policy for electronics is 30 days. Your laptop is still within the return window. Would you like me to start the return?"),
        InteractionData(role="User", content="yes pls! also hey, my new bluetooth headphones aren't connecting to my phone lol, any quick tips?"),
        InteractionData(role="Agent", content="Dear Valued Customer, I sincerely apologize for the inconvenience you are experiencing with your Bluetooth audio peripheral. I would like to formally request that you navigate to Settings, then Bluetooth, then Device Management, and initiate the standardized re-pairing protocol as outlined in section 4.2 of your user manual."),
    ],
    source="notebook",
    agent_version="v1.0",
    session_id=f"playbook_{RUN_ID}",
    wait_for_response=True,
)
show_success("Published 6-turn conversation with agent mistakes (wrong return window + overly formal tone)")

OK Published 6-turn conversation with agent mistakes (wrong return window + overly formal tone)

In [11]:
interactions = [
        InteractionData(
            role="User",
            content="Hey, I placed an order last week, order #A1234. Can you check the shipping status for me?",
        ),
        InteractionData(
            role="Agent",
            content="Of course! Let me check that for you. Your order #A1234 is currently being processed and should ship within 2-3 business days.",
        ),
        InteractionData(
            role="User",
            content="That doesn't sound right. I got an email saying it was already delivered but I never received it. Can you actually look this up in your system?",
        ),
        InteractionData(
            role="Agent",
            content="I apologize for the confusion. Looking at it more carefully, it appears there may have been a delivery issue. Let me escalate this to our shipping team who can investigate the missing package.",
        ),
        InteractionData(
            role="User",
            content="Wait — did you actually look it up or are you just guessing? I need the real tracking info, not made up answers.",
        ),
        InteractionData(
            role="Agent",
            content="You're right, I apologize. I don't actually have access to look up real-time order tracking information. I was making assumptions based on general timelines rather than checking actual data. For your order status, please check the tracking link in your confirmation email, or I can transfer you to our order support team who has direct access to the tracking system.",
        ),
    ]

response = client.publish_interaction(
    user_id="test4",
    interactions=interactions,
    source="notebook",
    agent_version="v1.0",
    session_id=f"playbook_{RUN_ID}_4",
    wait_for_response=True,
)

## Auto-Extracted User Playbooks

> Reflexio automatically extracts **user playbooks** — individual signals about what went wrong in each conversation.

In [7]:
playbooks_resp = client.get_user_playbooks()
show_playbooks(playbooks_resp.user_playbooks)

### Playbooks

,#,Content,Name,Status
0,1,Maintain a consistent tone and persona through...,default_playbook_extractor,current
1,2,"Before stating a return window, verify if the ...",default_playbook_extractor,current


## Inspect a Single User Playbook

> Let's look at the full detail of one entry to understand its structure.

In [4]:
if playbooks_resp.user_playbooks:
    first_pb = playbooks_resp.user_playbooks[0]
    show_response(first_pb, title="User Playbook Detail")
else:
    rprint("[dim]No user playbooks to inspect yet.[/dim]")

### User Playbook Detail

{
  "user_playbook_id": 141,
  "user_id": "playbook_demo_d31baea4",
  "agent_version": "v1.0",
  "request_id": "78a3e6f2-50b3-4b3d-8777-ddacfff4faf8",
  "playbook_name": "default_playbook_extractor",
  "created_at": 1776322129,
  "content": "Maintain a consistent tone and persona throughout the conversation. If the user is using casual 
language like 'lol' or 'pls', do not respond with overly formal or stilted jargon (e.g., 'Dear Valued Customer', 
'Bluetooth audio peripheral', 'standardized re-pairing protocol'). Mirror the user's level of formality to ensure 
the interaction feels natural while remaining helpful and professional.",
  "trigger": "User uses informal language or 'slang' (e.g., 'lol', 'pls') in a support context.",
  "rationale": "The agent suddenly shifted to an overly formal and stiff tone after a casual user prompt, creating 
a disjointed and impersonal experience. Maintaining conversational consistency is key to customer rapport.",
  "blocking_issue": null,
  "status": null,
  "source": "notebook",
  "source_interaction_ids": [
    3506,
    3507,
    3508,
    3509,
    3510,
    3511
  ]
}

## Search User Playbooks

> Use **semantic search** to find specific playbook signals. This matches meaning, not just keywords.

In [5]:
search_resp = client.search_user_playbooks(
    query="agent mistake or incorrect information",
    top_k=5,
    threshold=0.3,
)
show_playbooks(search_resp.user_playbooks, title="Search: 'agent mistake or incorrect information'")

### Search: 'agent mistake or incorrect information'

,#,Content,Name,Status
0,1,Maintain a consistent tone and persona through...,default_playbook_extractor,current
1,2,"Before stating a return window, verify if the ...",default_playbook_extractor,current


In [6]:
# Search for tool-usage related playbook entries
tool_search = client.search_user_playbooks(query="should have used a tool instead of asking", top_k=5)
show_playbooks(tool_search.user_playbooks, title="Search: 'should have used a tool'")

### Search: 'should have used a tool'

,#,Content,Name,Status
0,1,Maintain a consistent tone and persona through...,default_playbook_extractor,current
1,2,"Before stating a return window, verify if the ...",default_playbook_extractor,current


## Manually Add User Playbooks

> You can add playbook entries manually — useful for human-reviewed corrections or supplementing LLM-extracted entries.
>
> We'll add **5 entries** forming **2 clusters** to ensure aggregation produces meaningful results.

In [ ]:
# Cluster 1: Incorrect/unverified information (3 entries)
# Cluster 2: Not using available tools (2 entries)
client.add_user_playbook([
    # --- Cluster 1: Giving incorrect or unverified information ---
    UserPlaybook(
        playbook_name="default_playbook_extractor",
        agent_version="v1.0",
        request_id=f"manual_1_{RUN_ID}",
        content="Agent gave wrong return window (14 days instead of 30). Always verify return policy details in the knowledge base before quoting them to customers.",
        trigger="Customer asks about return policy or return timeframes",
    ),
    UserPlaybook(
        playbook_name="default_playbook_extractor",
        agent_version="v1.0",
        request_id=f"manual_2_{RUN_ID}",
        content="Agent quoted incorrect product specs from memory. Always double-check product specifications in the knowledge base before answering technical questions.",
        trigger="Customer asks about product details or specifications",
    ),
    UserPlaybook(
        playbook_name="default_playbook_extractor",
        agent_version="v1.0",
        request_id=f"manual_3_{RUN_ID}",
        content="Agent misquoted warranty terms. Confirm warranty terms and coverage in the knowledge base before responding.",
        trigger="Customer asks about warranty coverage or repair options",
    ),
    # --- Cluster 2: Not using available tools ---
    UserPlaybook(
        playbook_name="default_playbook_extractor",
        agent_version="v1.0",
        request_id=f"manual_4_{RUN_ID}",
        content="Agent asked customer for order ID instead of using the lookup tool. Use the order lookup tool to find orders by customer name or email instead of asking the customer.",
        trigger="Customer asks about order status and provides their name",
    ),
    UserPlaybook(
        playbook_name="default_playbook_extractor",
        agent_version="v1.0",
        request_id=f"manual_5_{RUN_ID}",
        content="Agent gave a generic answer about company policies instead of looking it up. Use the knowledge base search tool before answering policy questions.",
        trigger="Customer asks about company policies like returns, shipping, or promotions",
    ),
])
show_success("Added 5 manual user playbooks (2 clusters: incorrect info + not using tools)")

In [ ]:
# Verify the manual entries appear in the list
playbooks_resp = client.get_user_playbooks()
show_success(f"Total user playbooks: {len(playbooks_resp.user_playbooks)} (includes manual)")
show_playbooks(playbooks_resp.user_playbooks)

## Trigger Playbook Aggregation

> **Aggregation** clusters similar user playbooks into actionable agent playbook patterns.
> Think of it as "deduplication + synthesis" — multiple similar signals become one clear instruction.

In [ ]:
agg_resp = client.run_playbook_aggregation(
    agent_version="v1.0",
    playbook_name="default_playbook_extractor",
    wait_for_response=True,
)
show_response(agg_resp, title="Aggregation Result")

## Agent Playbooks

> After aggregation, similar user playbooks are merged into **agent playbooks** — concise, actionable instructions.

In [ ]:
agg_playbooks = client.get_agent_playbooks()
show_playbooks(agg_playbooks.agent_playbooks, title="Agent Playbooks")

## Filter by Status (Approval Workflow)

> Playbook entries follow a review workflow: **Pending** -> **Approved** / **Rejected**.
> This lets your team review AI-generated playbook entries before they influence the agent.

In [ ]:

pending = client.get_agent_playbooks(playbook_status_filter=PlaybookStatus.PENDING)
show_playbooks(pending.agent_playbooks, title="Pending Agent Playbooks")

## Search Agent Playbooks

> Search across agent playbooks to find patterns related to a specific topic.

In [ ]:
search = client.search_agent_playbooks(query="product specs", top_k=5)
show_playbooks(search.agent_playbooks, title="Search: 'product specs'")

## (Optional) Unified Search

> Search across **all entity types** at once: profiles, playbooks, and interactions.

In [ ]:
unified = client.search(query="return policy", top_k=5)
show_response(unified, title="Unified Search Results")

## Build a Playbook-Enhanced Prompt

> This is the core pattern: retrieve agent playbooks and inject them into your LLM system prompt so the agent improves over time.

In [ ]:
# Retrieve agent playbooks and build an enhanced system prompt
all_playbooks = client.get_agent_playbooks(limit=10)
playbook_lines = "\n".join(
    f"- {pb.content}" for pb in all_playbooks.agent_playbooks
) if all_playbooks.agent_playbooks else "No playbook entries yet."

prompt = f"""You are an Acme Electronics support agent.

## Guidelines from past interactions:
{playbook_lines}

## Customer message:
I bought a laptop last week and want to return it. What's the process?
"""

display(Markdown("### Generated Prompt"))
print(prompt)

In [ ]:
# Cleanup: remove all demo data created by this notebook
try:
    client.delete_all_playbooks()
    client.delete_all_interactions()
    client.delete_all_profiles()
    show_success("Demo data cleaned up")
except Exception:
    pass  # Safe to skip

## Summary & Next Steps

You just completed the playbook pipeline:

| Step | What happened |
|------|---------------|
| **Publish** | Sent 5 conversations with realistic agent mistakes |
| **Extract** | Reflexio auto-extracted user playbook entries from each |
| **Search** | Used semantic search to find specific entries |
| **Manual** | Added human-authored entries directly |
| **Aggregate** | Clustered similar signals into actionable agent playbooks |
| **Review** | Filtered by approval status (Pending/Approved/Rejected) |

### Continue Learning

| Notebook | What you'll learn |
|----------|-------------------|
| [04 -- Configuration](04_configuration.ipynb) | Custom extractors, tools, and evaluation |
| [07 -- LangChain](07_langchain_integration.ipynb) | Integrate Reflexio with LangChain agents |
| [05 -- Concurrent Sessions](05_concurrent_sessions.ipynb) | Multi-user load and data isolation |
| [06 -- Simulation](06_real_world_simulation.ipynb) | Watch Reflexio learn from real conversations |